# Script 12C — Corrected EWMA Alignment and Regime-Robust Volatility Validation

## Purpose

This notebook corrects and audits the volatility forecasting comparison before the research moves to drift/sensitivity analysis.

It will:

1. Load the **development-only** dataset and Script 12 candidate forecasts.
2. Recalculate the future realized-variance target independently and verify its date alignment.
3. Rebuild EWMA94 using returns observable **through the forecast-origin date**. The previous Script 12 implementation used the prior row's return in its update, leaving EWMA one observation behind.
4. Rebuild HIST20/HIST60 baselines using only returns observable at the forecast origin.
5. Recompute the robust anchor and stress ratio.
6. Re-select Script 12B strategies using corrected **previous-year** QLIKE only.
7. Evaluate corrected forecasts against the corrected EWMA benchmark on matched forecast origins.
8. Produce stability, VaR and quality-gate outputs.

### Important safeguards

- This notebook does **not** overwrite Script 12 or Script 12B source files.
- It does **not** fit new GARCH models; it reuses the candidate model forecasts produced by Script 12.
- The old Script 12B candidate-score file is used only to recover the strategy definitions. Its old `Inner_QLIKE` values are **not** used for selection.
- The 2024–2025 final holdout must remain untouched. This notebook stops if development forecasts include dates in 2024 or later.
- The robust volatility forecast is evaluated against **EWMA**, not the point-forecast Random Walk.
- Results are not approved automatically just because QLIKE is lower. Inspect all audit outputs and limitations.

In [ ]:
%pip install numpy pandas matplotlib scipy

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.8f}")

CURRENCIES = ["USD", "GBP", "EUR", "JPY"]
HORIZONS = [1, 3, 7]
OUTER_YEARS = [2019, 2020, 2021, 2022, 2023]
BASELINE_MODELS = ["EWMA94", "HIST20", "HIST60"]
GARCH_MODELS = ["GARCH_NORMAL", "GARCH_T", "EGARCH_T", "GJR_GARCH_T"]
REQUIRED_MODELS = GARCH_MODELS + BASELINE_MODELS
LAMBDA_EWMA = 0.94
EPS = 1e-10
TARGET_ATOL = 1e-8
TARGET_RTOL = 1e-6

print("Configuration loaded.")
print("HOLDOUT LOCK: dates in 2024–2025 must not be used for model selection.")

In [ ]:
def find_project_root():
    candidates = [Path.cwd()] + list(Path.cwd().parents)
    for candidate in candidates:
        if (candidate / "data" / "processed" / "02_development_model_dataset.csv").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find data/processed/02_development_model_dataset.csv. "
        "Save this notebook inside your Research project (preferably Research/notebooks/) "
        "and confirm Notebook 02 has completed."
    )

ROOT = find_project_root()
DEV_PATH = ROOT / "data" / "processed" / "02_development_model_dataset.csv"
S12_CANDIDATE_PATH = ROOT / "outputs" / "predictions" / "12_volatility_candidate_predictions_2018_2023.csv"
S12B_SCORE_PATH = ROOT / "outputs" / "tables" / "12B_regime_robust_volatility_candidate_scores.csv"

PRED_DIR = ROOT / "outputs" / "predictions"
TABLE_DIR = ROOT / "outputs" / "tables"
FIGURE_DIR = ROOT / "outputs" / "figures"
for folder in [PRED_DIR, TABLE_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Development data:", DEV_PATH)
print("Script 12 candidate forecasts:", S12_CANDIDATE_PATH)
print("Script 12B strategy definitions:", S12B_SCORE_PATH)

missing = [str(p) for p in [DEV_PATH, S12_CANDIDATE_PATH, S12B_SCORE_PATH] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required input file(s) not found:\n- " + "\n- ".join(missing) +
        "\nRun Notebook 02, then Script 12, then the original Script 12B once to create its strategy-definition file. "
        "Do not run the final 2024–2025 holdout."
    )

In [ ]:
df = pd.read_csv(DEV_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
candidate_source = pd.read_csv(S12_CANDIDATE_PATH, parse_dates=["Forecast_Origin"])
old_score_source = pd.read_csv(S12B_SCORE_PATH)

required_data_cols = ["Date"] + [f"{cur}_mid" for cur in CURRENCIES]
required_candidate_cols = [
    "Currency", "Horizon", "Year", "Forecast_Origin",
    "Realized_Variance", "Model", "Variance_Forecast"
]
required_score_cols = [
    "Strategy", "Model", "Model_Weight", "Stress_Threshold", "Stress_Cap"
]

for col in required_data_cols:
    if col not in df.columns:
        raise KeyError(f"Development dataset is missing required column: {col}")
for col in required_candidate_cols:
    if col not in candidate_source.columns:
        raise KeyError(f"Script 12 candidate file is missing required column: {col}")
for col in required_score_cols:
    if col not in old_score_source.columns:
        raise KeyError(f"Script 12B score file is missing required column: {col}")

df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()
candidate_source["Forecast_Origin"] = pd.to_datetime(candidate_source["Forecast_Origin"]).dt.normalize()
candidate_source["Year"] = pd.to_numeric(candidate_source["Year"], errors="raise").astype(int)
candidate_source["Horizon"] = pd.to_numeric(candidate_source["Horizon"], errors="raise").astype(int)
candidate_source["Realized_Variance"] = pd.to_numeric(candidate_source["Realized_Variance"], errors="coerce")
candidate_source["Variance_Forecast"] = pd.to_numeric(candidate_source["Variance_Forecast"], errors="coerce")

assert df["Date"].is_monotonic_increasing, "STOP: development dataset is not sorted by Date."
assert not df["Date"].duplicated().any(), "STOP: duplicate dates in development dataset."
assert df["Date"].max() < pd.Timestamp("2024-01-01"), "STOP: development dataset includes the final holdout."
assert candidate_source["Forecast_Origin"].max() < pd.Timestamp("2024-01-01"), "STOP: Script 12 candidate forecasts include 2024 or later."
assert set(candidate_source["Currency"].unique()).issubset(set(CURRENCIES)), "Unexpected currency in Script 12 candidates."
assert set(candidate_source["Horizon"].unique()).issubset(set(HORIZONS)), "Unexpected horizon in Script 12 candidates."

print("Development range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Development rows:", len(df))
print("Candidate forecast rows:", len(candidate_source))
print("Candidate forecast years:", sorted(candidate_source["Year"].unique().tolist()))
print("PASS — 2024–2025 remains outside this notebook.")
display(candidate_source.head())

## Step 1 — Independently rebuild returns and future realized variance

The source code for Script 12 defines returns as `100 × log(mid_t / mid_(t-1))`. The target at origin \(t\) for horizon \(h\) is the sum of squared returns on the **next** \(h\) observed rows:

\[
RV_{t,h}=\sum_{j=1}^{h}r_{t+j}^{2}.
\]

This target is not allowed to include the return at \(t\) itself. The return at \(t\), however, is available to a forecast made after the quote for date \(t\) has been observed.

In [ ]:
series = {}
target_parts = []
baseline_parts = []
stress_parts = []

def ewma_variance_available_at_origin(returns, lam=0.94, seed_window=20):
    # At origin t, use observed squared returns through t, including r_t.
    # Early estimates use an expanding mean; subsequent estimates use the EWMA recursion.
    r = np.asarray(returns, dtype=float)
    r2 = r ** 2
    out = np.full(len(r), np.nan, dtype=float)

    for i in range(len(r)):
        if not np.isfinite(r2[i]):
            out[i] = out[i - 1] if i > 0 else np.nan
            continue

        if i < seed_window:
            hist = r2[:i + 1]
            hist = hist[np.isfinite(hist)]
            out[i] = float(np.mean(hist)) if len(hist) else np.nan
        else:
            previous = out[i - 1] if i > 0 else np.nan
            if not np.isfinite(previous):
                hist = r2[max(0, i - seed_window):i]
                hist = hist[np.isfinite(hist)]
                previous = float(np.mean(hist)) if len(hist) else np.nan
            out[i] = (
                lam * previous + (1.0 - lam) * r2[i]
                if np.isfinite(previous) else np.nan
            )
    return out

for cur in CURRENCIES:
    mid = pd.to_numeric(df[f"{cur}_mid"], errors="coerce")
    ret = 100.0 * np.log(mid / mid.shift(1))
    r2 = ret ** 2

    temp = pd.DataFrame({
        "Date": df["Date"],
        "Mid": mid,
        "Return": ret,
        "Return_Squared": r2
    })

    for h in HORIZONS:
        rv = pd.Series(0.0, index=temp.index, dtype=float)
        for j in range(1, h + 1):
            rv = rv + r2.shift(-j)
        temp[f"RV_H{h}"] = rv

    temp["EWMA_Daily_Corrected"] = ewma_variance_available_at_origin(
        ret.to_numpy(), lam=LAMBDA_EWMA, seed_window=20
    )
    temp["HIST20_Daily"] = r2.rolling(20, min_periods=10).mean()
    temp["HIST60_Daily"] = r2.rolling(60, min_periods=20).mean()

    # Both rolling windows include only returns observable through the forecast origin.
    var10 = r2.rolling(10, min_periods=5).mean()
    var60 = r2.rolling(60, min_periods=30).mean()
    temp["Stress_Ratio"] = var10 / var60.replace(0, np.nan)

    series[cur] = temp

    for h in HORIZONS:
        target_parts.append(pd.DataFrame({
            "Currency": cur,
            "Horizon": h,
            "Forecast_Origin": temp["Date"],
            "Year": temp["Date"].dt.year,
            "Realized_Variance_Recomputed": temp[f"RV_H{h}"]
        }))

        baseline_parts.append(pd.DataFrame({
            "Currency": cur,
            "Horizon": h,
            "Forecast_Origin": temp["Date"],
            "Year": temp["Date"].dt.year,
            "Realized_Variance": temp[f"RV_H{h}"],
            "EWMA94": h * temp["EWMA_Daily_Corrected"],
            "HIST20": h * temp["HIST20_Daily"],
            "HIST60": h * temp["HIST60_Daily"]
        }))

    stress_parts.append(pd.DataFrame({
        "Currency": cur,
        "Forecast_Origin": temp["Date"],
        "Stress_Ratio": temp["Stress_Ratio"]
    }))

target_lookup = pd.concat(target_parts, ignore_index=True)
baseline_wide = pd.concat(baseline_parts, ignore_index=True)
stress_lookup = pd.concat(stress_parts, ignore_index=True)

print("Independent targets and baselines created.")
display(baseline_wide.query("Currency == 'USD' and Horizon == 1").head())

## Step 2 — Target alignment and EWMA timing audit

The target audit must pass before any corrected scores are accepted. The EWMA audit then compares the old Script 12 EWMA values with the origin-aligned values reconstructed above. The old source file is preserved.

In [ ]:
candidate_target_audit = candidate_source.merge(
    target_lookup,
    on=["Currency", "Horizon", "Forecast_Origin"],
    how="left",
    validate="many_to_one"
)
candidate_target_audit["Target_Abs_Difference"] = (
    candidate_target_audit["Realized_Variance"]
    - candidate_target_audit["Realized_Variance_Recomputed"]
).abs()
candidate_target_audit["Target_Matches"] = np.isclose(
    candidate_target_audit["Realized_Variance"],
    candidate_target_audit["Realized_Variance_Recomputed"],
    atol=TARGET_ATOL,
    rtol=TARGET_RTOL,
    equal_nan=False
)

target_alignment_summary = (
    candidate_target_audit.groupby(["Currency", "Horizon"], as_index=False)
    .agg(
        Candidate_Rows=("Target_Matches", "size"),
        Target_Matches=("Target_Matches", "sum"),
        Max_Abs_Target_Difference=("Target_Abs_Difference", "max"),
        Median_Abs_Target_Difference=("Target_Abs_Difference", "median")
    )
)
target_alignment_summary["Mismatches"] = (
    target_alignment_summary["Candidate_Rows"] - target_alignment_summary["Target_Matches"]
)
display(target_alignment_summary)

n_target_mismatches = int((~candidate_target_audit["Target_Matches"]).sum())
if n_target_mismatches:
    candidate_target_audit.loc[~candidate_target_audit["Target_Matches"]].head(20).to_csv(
        TABLE_DIR / "12C_target_alignment_mismatches.csv", index=False
    )
    raise ValueError(
        f"STOP: {n_target_mismatches} Script 12 candidate rows do not match independently "
        "recomputed future realized variance. Do not report model scores until this is resolved."
    )
print("PASS — candidate realized variance matches the independently rebuilt future target.")

old_ewma = candidate_source.loc[
    candidate_source["Model"].eq("EWMA94"),
    ["Currency", "Horizon", "Forecast_Origin", "Variance_Forecast"]
].rename(columns={"Variance_Forecast": "EWMA94_Old"})

correct_ewma = baseline_wide[
    ["Currency", "Horizon", "Forecast_Origin", "EWMA94"]
].rename(columns={"EWMA94": "EWMA94_Corrected"})

ewma_alignment_audit = old_ewma.merge(
    correct_ewma,
    on=["Currency", "Horizon", "Forecast_Origin"],
    how="inner",
    validate="one_to_one"
)
ewma_alignment_audit["Absolute_Difference"] = (
    ewma_alignment_audit["EWMA94_Corrected"] - ewma_alignment_audit["EWMA94_Old"]
).abs()
ewma_alignment_audit["Relative_Difference"] = (
    ewma_alignment_audit["Absolute_Difference"]
    / ewma_alignment_audit["EWMA94_Corrected"].abs().clip(lower=EPS)
)

print("Old vs corrected EWMA comparison:")
display(
    ewma_alignment_audit[
        ewma_alignment_audit["Forecast_Origin"].between("2022-03-01", "2022-03-18")
        & ewma_alignment_audit["Currency"].eq("USD")
        & ewma_alignment_audit["Horizon"].eq(1)
    ].head(30)
)
print(
    "Rows where EWMA changed:",
    int((ewma_alignment_audit["Absolute_Difference"] > 1e-12).sum()),
    "of", len(ewma_alignment_audit)
)
ewma_alignment_audit.to_csv(TABLE_DIR / "12C_ewma_alignment_audit.csv", index=False)
target_alignment_summary.to_csv(TABLE_DIR / "12C_target_alignment_summary.csv", index=False)

## Step 3 — Create corrected candidate forecasts without overwriting Script 12

The GARCH/EGARCH candidate forecasts are reused as generated by Script 12. Only the baseline forecasts are rebuilt. All candidate rows receive the independently reconstructed realized-variance target after the alignment audit has passed.

In [ ]:
candidate = candidate_source.copy()

candidate = candidate.drop(columns=["Realized_Variance"]).merge(
    target_lookup[
        ["Currency", "Horizon", "Forecast_Origin", "Year", "Realized_Variance_Recomputed"]
    ],
    on=["Currency", "Horizon", "Forecast_Origin", "Year"],
    how="left",
    validate="many_to_one"
).rename(columns={"Realized_Variance_Recomputed": "Realized_Variance"})

candidate_nonbaseline = candidate[~candidate["Model"].isin(BASELINE_MODELS)].copy()

baseline_long = baseline_wide.melt(
    id_vars=["Currency", "Horizon", "Forecast_Origin", "Year", "Realized_Variance"],
    value_vars=BASELINE_MODELS,
    var_name="Model",
    value_name="Variance_Forecast"
)
baseline_long = baseline_long[
    baseline_long["Year"].between(2018, 2023)
    & baseline_long["Realized_Variance"].notna()
    & baseline_long["Variance_Forecast"].notna()
    & np.isfinite(baseline_long["Variance_Forecast"])
    & (baseline_long["Variance_Forecast"] > 0)
].copy()

candidate_corrected = pd.concat(
    [
        candidate_nonbaseline[
            ["Currency", "Horizon", "Year", "Forecast_Origin", "Realized_Variance", "Model", "Variance_Forecast"]
        ],
        baseline_long[
            ["Currency", "Horizon", "Year", "Forecast_Origin", "Realized_Variance", "Model", "Variance_Forecast"]
        ]
    ],
    ignore_index=True
)

candidate_corrected = candidate_corrected[
    candidate_corrected["Year"].between(2018, 2023)
    & candidate_corrected["Forecast_Origin"].lt(pd.Timestamp("2024-01-01"))
    & np.isfinite(candidate_corrected["Realized_Variance"])
    & np.isfinite(candidate_corrected["Variance_Forecast"])
    & (candidate_corrected["Variance_Forecast"] > 0)
].copy()

assert candidate_corrected["Forecast_Origin"].max() < pd.Timestamp("2024-01-01")
assert set(BASELINE_MODELS).issubset(set(candidate_corrected["Model"].unique()))
assert set(GARCH_MODELS).issubset(set(candidate_corrected["Model"].unique()))

corrected_candidate_path = PRED_DIR / "12C_corrected_candidate_predictions_2018_2023.csv"
candidate_corrected.to_csv(corrected_candidate_path, index=False)

print("Saved corrected candidate forecasts:", corrected_candidate_path)
print("Corrected candidate rows:", len(candidate_corrected))
display(candidate_corrected.groupby(["Currency", "Horizon", "Model"]).size().rename("Rows").reset_index().head(30))

## Step 4 — Construct a fair matched-origin comparison panel

To compare strategy choices fairly, this notebook uses the same forecast origins for all four GARCH-family candidates and all three baselines within each currency, horizon and year. Rows missing any required candidate are excluded from the strategy tournament and counted in the coverage audit.

In [ ]:
wide = candidate_corrected.pivot_table(
    index=["Currency", "Horizon", "Year", "Forecast_Origin", "Realized_Variance"],
    columns="Model",
    values="Variance_Forecast",
    aggfunc="first"
).reset_index()
wide.columns.name = None

missing_model_columns = [m for m in REQUIRED_MODELS if m not in wide.columns]
if missing_model_columns:
    raise ValueError(f"STOP: corrected candidate panel lacks model columns: {missing_model_columns}")

coverage_rows = []
for (cur, h, year), g in wide.groupby(["Currency", "Horizon", "Year"]):
    n_all = len(g)
    complete = g.dropna(subset=REQUIRED_MODELS)
    coverage_rows.append({
        "Currency": cur,
        "Horizon": int(h),
        "Year": int(year),
        "Rows_Available_Any_Model": int(n_all),
        "Rows_Matched_All_Models": int(len(complete)),
        "Rows_Excluded_For_Missing_Candidate": int(n_all - len(complete)),
        "Matched_Percent": 100.0 * len(complete) / n_all if n_all else np.nan
    })
coverage_audit = pd.DataFrame(coverage_rows)
display(coverage_audit.head(30))
coverage_audit.to_csv(TABLE_DIR / "12C_matched_origin_coverage.csv", index=False)

wide = wide.dropna(subset=REQUIRED_MODELS).copy()
wide["Robust_Anchor"] = wide[BASELINE_MODELS].median(axis=1)

wide = wide.merge(
    stress_lookup,
    on=["Currency", "Forecast_Origin"],
    how="left",
    validate="many_to_one"
)

wide = wide[
    np.isfinite(wide["Realized_Variance"])
    & np.isfinite(wide["Robust_Anchor"])
    & (wide["Robust_Anchor"] > 0)
    & np.isfinite(wide["Stress_Ratio"])
].copy()

if wide.empty:
    raise RuntimeError("No complete matched forecast origins remain. Inspect the coverage audit.")
if (wide["Forecast_Origin"] >= pd.Timestamp("2024-01-01")).any():
    raise ValueError("STOP: final holdout dates entered the matched panel.")

print("Matched panel rows:", len(wide))
print("Matched panel date range:", wide["Forecast_Origin"].min().date(), "to", wide["Forecast_Origin"].max().date())
display(wide.head())

## Step 5 — Recover the tested strategy definitions

The previous Script 12B score file supplies the tested strategy configurations only. The notebook deliberately ignores its old `Inner_QLIKE` values and re-evaluates every configuration using the corrected baselines, corrected robust anchor and corrected stress ratio.

In [ ]:
config_cols = ["Strategy", "Model", "Model_Weight", "Stress_Threshold", "Stress_Cap"]
configs = old_score_source[config_cols].drop_duplicates().copy()

config_consistency = (
    old_score_source.groupby("Strategy")[["Model", "Model_Weight", "Stress_Threshold", "Stress_Cap"]]
    .nunique(dropna=False)
)
if (config_consistency > 1).any().any():
    raise ValueError("STOP: a strategy name maps to inconsistent parameters in the Script 12B score file.")

configs["Model_Weight"] = pd.to_numeric(configs["Model_Weight"], errors="coerce")
configs["Stress_Threshold"] = pd.to_numeric(configs["Stress_Threshold"], errors="coerce")
configs["Stress_Cap"] = pd.to_numeric(configs["Stress_Cap"], errors="coerce")

unknown_models = sorted(set(configs["Model"]) - set(GARCH_MODELS) - {"Robust_Anchor"})
if unknown_models:
    raise ValueError(f"STOP: unsupported strategy model(s) found: {unknown_models}")

if configs["Strategy"].duplicated().any():
    raise ValueError("STOP: duplicate strategy definitions remain after de-duplication.")
if configs.empty:
    raise ValueError("STOP: no strategy definitions were found.")

print("Strategy definitions to re-evaluate:", len(configs))
display(configs.sort_values("Strategy").head(30))

In [ ]:
def qlike_losses(realized_variance, forecast_variance):
    rv = np.asarray(realized_variance, dtype=float)
    vf = np.asarray(forecast_variance, dtype=float)
    valid = np.isfinite(rv) & np.isfinite(vf) & (rv >= 0) & (vf > 0)
    out = np.full(len(rv), np.nan, dtype=float)
    rv_safe = np.maximum(rv[valid], EPS)
    vf_safe = np.maximum(vf[valid], EPS)
    out[valid] = np.log(vf_safe) + rv_safe / vf_safe
    return out

def qlike_score(rv, vf):
    loss = qlike_losses(rv, vf)
    loss = loss[np.isfinite(loss)]
    return float(np.mean(loss)) if len(loss) else np.nan

def apply_strategy(frame, config):
    # Robust forecast = effective weight * clipped selected-model forecast
    #                   + (1 - effective weight) * robust anchor.
    # In a stress regime, the model weight is capped by Stress_Cap.
    # GARCH forecasts are clipped to [0.1x, 10x] the robust anchor.
    g = frame.copy()
    anchor = g["Robust_Anchor"].to_numpy(dtype=float)
    model = config["Model"]

    if model == "Robust_Anchor":
        raw = anchor.copy()
        clipped = anchor.copy()
        effective_weight = np.zeros(len(g), dtype=float)
        forecast = anchor.copy()
        guardrail = np.zeros(len(g), dtype=bool)
        stress_active = np.zeros(len(g), dtype=bool)
    else:
        raw = g[model].to_numpy(dtype=float)
        clipped = np.clip(raw, 0.1 * anchor, 10.0 * anchor)
        guardrail = ~np.isclose(raw, clipped, rtol=1e-12, atol=1e-12)
        threshold = float(config["Stress_Threshold"])
        cap = float(config["Stress_Cap"])
        nominal_weight = float(config["Model_Weight"])
        stress_active = g["Stress_Ratio"].to_numpy(dtype=float) > threshold
        effective_weight = np.where(
            stress_active,
            np.minimum(nominal_weight, cap),
            nominal_weight
        )
        forecast = effective_weight * clipped + (1.0 - effective_weight) * anchor

    forecast = np.maximum(forecast, EPS)
    return pd.DataFrame({
        "Forecast": forecast,
        "Raw_Selected_Model_Forecast": raw,
        "Clipped_Selected_Model_Forecast": clipped,
        "Effective_Model_Weight": effective_weight,
        "Stress_Active": stress_active,
        "Guardrail_Active": guardrail
    }, index=g.index)

print("Forecast and scoring functions defined.")

## Step 6 — Re-select strategies using previous-year data only

For each currency, horizon and outer validation year:

- The strategy is selected on the immediately preceding year.
- All strategy candidates are scored on the same complete matched origins for that inner year.
- The selected configuration is then applied to the outer year.
- The outer-year outcomes do not influence that year's selection.

The inner years are 2018–2022; the outer validation years are 2019–2023.

In [ ]:
selection_rows = []
strategy_score_parts = []
prediction_parts = []

for cur in CURRENCIES:
    for h in HORIZONS:
        for outer_year in OUTER_YEARS:
            inner_year = outer_year - 1

            inner = wide[
                wide["Currency"].eq(cur)
                & wide["Horizon"].eq(h)
                & wide["Year"].eq(inner_year)
            ].copy()

            outer = wide[
                wide["Currency"].eq(cur)
                & wide["Horizon"].eq(h)
                & wide["Year"].eq(outer_year)
            ].copy()

            if len(inner) < 30:
                raise RuntimeError(
                    f"STOP: only {len(inner)} matched inner observations for {cur} H{h} "
                    f"in {inner_year}; insufficient for reliable strategy selection."
                )
            if len(outer) < 30:
                raise RuntimeError(
                    f"STOP: only {len(outer)} matched outer observations for {cur} H{h} "
                    f"in {outer_year}; inspect candidate coverage."
                )

            scores_this_task = []
            for _, config in configs.iterrows():
                applied = apply_strategy(inner, config)
                score = qlike_score(inner["Realized_Variance"], applied["Forecast"])
                if np.isfinite(score):
                    scores_this_task.append({
                        "Currency": cur,
                        "Horizon": h,
                        "Inner_Year": inner_year,
                        "Outer_Year": outer_year,
                        "Strategy": config["Strategy"],
                        "Model": config["Model"],
                        "Model_Weight": config["Model_Weight"],
                        "Stress_Threshold": config["Stress_Threshold"],
                        "Stress_Cap": config["Stress_Cap"],
                        "Inner_QLIKE_Corrected": score,
                        "Inner_Observations": len(inner)
                    })

            score_df = pd.DataFrame(scores_this_task)
            if score_df.empty:
                raise RuntimeError(f"No strategy could be scored for {cur} H{h}, inner year {inner_year}.")
            score_df = score_df.sort_values(
                ["Inner_QLIKE_Corrected", "Strategy"], ascending=[True, True]
            ).reset_index(drop=True)
            strategy_score_parts.append(score_df)

            best = score_df.iloc[0]
            config = configs.loc[configs["Strategy"].eq(best["Strategy"])].iloc[0]
            applied_outer = apply_strategy(outer, config)

            out = outer[
                ["Currency", "Horizon", "Year", "Forecast_Origin", "Realized_Variance",
                 "EWMA94", "Robust_Anchor", "Stress_Ratio"]
            ].copy()
            out = out.rename(columns={
                "Year": "Validation_Year",
                "EWMA94": "EWMA_Benchmark"
            })
            out["Regime_Robust_Forecast"] = applied_outer["Forecast"].to_numpy()
            out["Raw_Selected_Model_Forecast"] = applied_outer["Raw_Selected_Model_Forecast"].to_numpy()
            out["Clipped_Selected_Model_Forecast"] = applied_outer["Clipped_Selected_Model_Forecast"].to_numpy()
            out["Effective_Model_Weight"] = applied_outer["Effective_Model_Weight"].to_numpy()
            out["Stress_Active"] = applied_outer["Stress_Active"].to_numpy()
            out["Guardrail_Active"] = applied_outer["Guardrail_Active"].to_numpy()
            out["Selected_Strategy"] = best["Strategy"]
            out["Selected_Model"] = best["Model"]
            out["Inner_Year_Used"] = inner_year
            out["Inner_QLIKE_Corrected"] = best["Inner_QLIKE_Corrected"]

            prediction_parts.append(out)
            selection_rows.append({
                "Currency": cur,
                "Horizon": h,
                "Outer_Year": outer_year,
                "Inner_Year": inner_year,
                "Selected_Strategy": best["Strategy"],
                "Selected_Model": best["Model"],
                "Selected_Model_Weight": best["Model_Weight"],
                "Stress_Threshold": best["Stress_Threshold"],
                "Stress_Cap": best["Stress_Cap"],
                "Inner_QLIKE_Corrected": best["Inner_QLIKE_Corrected"],
                "Inner_Observations": best["Inner_Observations"],
                "Number_Of_Strategies_Evaluated": len(score_df),
                "Outer_Observations": len(outer)
            })

selections_corrected = pd.DataFrame(selection_rows)
strategy_scores_corrected = pd.concat(strategy_score_parts, ignore_index=True)
predictions_corrected = pd.concat(prediction_parts, ignore_index=True)

assert (selections_corrected["Inner_Year"] < selections_corrected["Outer_Year"]).all()
assert predictions_corrected["Forecast_Origin"].max() < pd.Timestamp("2024-01-01")
assert np.isfinite(predictions_corrected["Regime_Robust_Forecast"]).all()
assert (predictions_corrected["Regime_Robust_Forecast"] > 0).all()
assert np.isfinite(predictions_corrected["EWMA_Benchmark"]).all()
assert (predictions_corrected["EWMA_Benchmark"] > 0).all()

display(selections_corrected)
print("Corrected outer predictions:", len(predictions_corrected))

In [ ]:
corrected_pred_path = PRED_DIR / "12C_corrected_regime_robust_volatility_predictions_2019_2023.csv"
corrected_selection_path = TABLE_DIR / "12C_corrected_regime_robust_volatility_selections.csv"
corrected_scores_path = TABLE_DIR / "12C_corrected_regime_robust_volatility_strategy_scores.csv"

predictions_corrected.to_csv(corrected_pred_path, index=False)
selections_corrected.to_csv(corrected_selection_path, index=False)
strategy_scores_corrected.to_csv(corrected_scores_path, index=False)

print("Saved:", corrected_pred_path)
print("Saved:", corrected_selection_path)
print("Saved:", corrected_scores_path)

## Step 7 — Pooled and yearly performance against corrected EWMA

Primary metric: QLIKE, where lower is better. Secondary metrics: variance MSE and volatility RMSE. The Diebold–Mariano-style comparison uses a Bartlett/Newey–West variance estimate with lags up to \(h-1\) to account for overlapping horizon targets. Treat p-values as supporting evidence, not as proof by themselves.

In [ ]:
def variance_mse(rv, vf):
    rv = np.asarray(rv, dtype=float)
    vf = np.asarray(vf, dtype=float)
    m = np.isfinite(rv) & np.isfinite(vf)
    return float(np.mean((rv[m] - vf[m]) ** 2)) if m.any() else np.nan

def volatility_rmse(rv, vf):
    rv = np.asarray(rv, dtype=float)
    vf = np.asarray(vf, dtype=float)
    m = np.isfinite(rv) & np.isfinite(vf) & (rv >= 0) & (vf >= 0)
    if not m.any():
        return np.nan
    return float(np.sqrt(np.mean((np.sqrt(rv[m]) - np.sqrt(vf[m])) ** 2)))

def dm_qlike_hac(rv, model_v, benchmark_v, horizon=1):
    rv = np.asarray(rv, dtype=float)
    model_v = np.asarray(model_v, dtype=float)
    benchmark_v = np.asarray(benchmark_v, dtype=float)
    valid = (
        np.isfinite(rv) & np.isfinite(model_v) & np.isfinite(benchmark_v)
        & (rv >= 0) & (model_v > 0) & (benchmark_v > 0)
    )
    rv, model_v, benchmark_v = rv[valid], model_v[valid], benchmark_v[valid]
    n = len(rv)
    if n < 30:
        return np.nan, np.nan, np.nan

    d = qlike_losses(rv, benchmark_v) - qlike_losses(rv, model_v)
    d = d[np.isfinite(d)]
    n = len(d)
    if n < 30:
        return np.nan, np.nan, np.nan

    mean_d = float(np.mean(d))
    centered = d - mean_d
    max_lag = min(max(0, int(horizon) - 1), n - 2)
    long_run_var = float(np.dot(centered, centered) / n)

    for lag in range(1, max_lag + 1):
        gamma = float(np.dot(centered[lag:], centered[:-lag]) / n)
        weight = 1.0 - lag / (max_lag + 1.0)
        long_run_var += 2.0 * weight * gamma

    if not np.isfinite(long_run_var) or long_run_var <= 0:
        return np.nan, np.nan, mean_d

    stat = mean_d / np.sqrt(long_run_var / n)
    p_value = 2.0 * stats.t.sf(abs(stat), df=n - 1)
    return float(stat), float(p_value), mean_d

pooled_rows = []
for (cur, h), g in predictions_corrected.groupby(["Currency", "Horizon"]):
    rv = g["Realized_Variance"].to_numpy(float)
    mv = g["Regime_Robust_Forecast"].to_numpy(float)
    bv = g["EWMA_Benchmark"].to_numpy(float)

    model_q = qlike_score(rv, mv)
    bench_q = qlike_score(rv, bv)
    model_mse = variance_mse(rv, mv)
    bench_mse = variance_mse(rv, bv)
    dm_stat, dm_p, qlike_gain = dm_qlike_hac(rv, mv, bv, horizon=h)

    pooled_rows.append({
        "Currency": cur,
        "Horizon": h,
        "Observations": len(g),
        "EWMA_QLIKE": bench_q,
        "Corrected_QLIKE": model_q,
        "QLIKE_Gain_vs_EWMA": bench_q - model_q,
        "EWMA_Variance_MSE": bench_mse,
        "Corrected_Variance_MSE": model_mse,
        "Variance_MSE_Skill_vs_EWMA": 1 - model_mse / bench_mse if bench_mse > 0 else np.nan,
        "EWMA_Vol_RMSE": volatility_rmse(rv, bv),
        "Corrected_Vol_RMSE": volatility_rmse(rv, mv),
        "DM_Stat_QLIKE": dm_stat,
        "DM_pvalue_QLIKE": dm_p,
        "Beats_EWMA_QLIKE": bool(model_q < bench_q),
        "Significant_QLIKE_Improvement_5pct": bool((model_q < bench_q) and np.isfinite(dm_p) and dm_p < 0.05)
    })

pooled_results = pd.DataFrame(pooled_rows).sort_values(["Currency", "Horizon"])
display(pooled_results)

yearly_rows = []
for (cur, h, year), g in predictions_corrected.groupby(["Currency", "Horizon", "Validation_Year"]):
    rv = g["Realized_Variance"].to_numpy(float)
    mv = g["Regime_Robust_Forecast"].to_numpy(float)
    bv = g["EWMA_Benchmark"].to_numpy(float)
    dm_stat, dm_p, qlike_gain = dm_qlike_hac(rv, mv, bv, horizon=h)
    model_q = qlike_score(rv, mv)
    bench_q = qlike_score(rv, bv)

    yearly_rows.append({
        "Currency": cur,
        "Horizon": h,
        "Validation_Year": int(year),
        "Observations": len(g),
        "EWMA_QLIKE": bench_q,
        "Corrected_QLIKE": model_q,
        "QLIKE_Gain_vs_EWMA": bench_q - model_q,
        "EWMA_Variance_MSE": variance_mse(rv, bv),
        "Corrected_Variance_MSE": variance_mse(rv, mv),
        "EWMA_Vol_RMSE": volatility_rmse(rv, bv),
        "Corrected_Vol_RMSE": volatility_rmse(rv, mv),
        "DM_Stat_QLIKE": dm_stat,
        "DM_pvalue_QLIKE": dm_p,
        "Beats_EWMA_QLIKE": bool(model_q < bench_q)
    })

yearly_results = pd.DataFrame(yearly_rows).sort_values(
    ["Currency", "Horizon", "Validation_Year"]
)
display(yearly_results.head(40))

print(
    "Pooled QLIKE wins:",
    int(pooled_results["Beats_EWMA_QLIKE"].sum()),
    "/", len(pooled_results)
)
print(
    "Pooled statistically significant QLIKE improvements:",
    int(pooled_results["Significant_QLIKE_Improvement_5pct"].sum()),
    "/", len(pooled_results)
)

In [ ]:
pooled_results.to_csv(TABLE_DIR / "12C_corrected_volatility_pooled_results.csv", index=False)
yearly_results.to_csv(TABLE_DIR / "12C_corrected_volatility_yearly_results.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 5))
plot_data = pooled_results.copy()
plot_data["Task"] = plot_data["Currency"] + " H" + plot_data["Horizon"].astype(str)
x = np.arange(len(plot_data))
ax.bar(x - 0.18, plot_data["EWMA_QLIKE"], width=0.36, label="Corrected EWMA")
ax.bar(x + 0.18, plot_data["Corrected_QLIKE"], width=0.36, label="Corrected Script 12B")
ax.set_xticks(x)
ax.set_xticklabels(plot_data["Task"], rotation=45, ha="right")
ax.set_ylabel("QLIKE (lower is better)")
ax.set_title("Script 12C — Pooled Volatility Forecast Comparison")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "12C_corrected_volatility_qlike_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

## Step 8 — H1 95% VaR exceedance and Kupiec unconditional-coverage test

This is a secondary diagnostic, not the primary volatility score. The forecast variance and future return are both in percentage-log-return units. A Normal lower-tail 5% threshold is used here as a transparent baseline; it does not claim that return tails are Normal.

In [ ]:
def kupiec_uc_test(exceedances, expected_rate=0.05):
    exc = np.asarray(exceedances, dtype=bool)
    n = len(exc)
    x = int(exc.sum())
    if n == 0:
        return {"N": 0, "Exceedances": 0, "Expected_Rate": expected_rate,
                "Observed_Rate": np.nan, "LR_UC": np.nan, "p_value": np.nan}

    phat = x / n
    p0 = float(expected_rate)

    def xlogy_count(count, prob):
        if count == 0:
            return 0.0
        if prob <= 0:
            return -np.inf
        return count * np.log(prob)

    ll_null = xlogy_count(x, p0) + xlogy_count(n - x, 1 - p0)
    if phat in (0.0, 1.0):
        ll_alt = 0.0
    else:
        ll_alt = xlogy_count(x, phat) + xlogy_count(n - x, 1 - phat)

    lr = max(0.0, 2.0 * (ll_alt - ll_null))
    p_value = float(stats.chi2.sf(lr, df=1))
    return {
        "N": n,
        "Exceedances": x,
        "Expected_Rate": p0,
        "Observed_Rate": phat,
        "LR_UC": lr,
        "p_value": p_value
    }

var_rows = []
z05 = float(stats.norm.ppf(0.05))

for cur in CURRENCIES:
    g = predictions_corrected[
        predictions_corrected["Currency"].eq(cur)
        & predictions_corrected["Horizon"].eq(1)
    ].copy()

    tmp = series[cur][["Date", "Return"]].copy()
    tmp["Next_Return"] = tmp["Return"].shift(-1)
    tmp = tmp.rename(columns={"Date": "Forecast_Origin"})

    g = g.merge(
        tmp[["Forecast_Origin", "Next_Return"]],
        on="Forecast_Origin",
        how="left",
        validate="one_to_one"
    )
    g["Sigma_Forecast"] = np.sqrt(g["Regime_Robust_Forecast"].clip(lower=EPS))
    g["VaR_5pct_Threshold"] = z05 * g["Sigma_Forecast"]

    valid = (
        np.isfinite(g["Next_Return"])
        & np.isfinite(g["VaR_5pct_Threshold"])
        & np.isfinite(g["Sigma_Forecast"])
    )
    exc = (g.loc[valid, "Next_Return"] < g.loc[valid, "VaR_5pct_Threshold"]).to_numpy()
    test = kupiec_uc_test(exc, expected_rate=0.05)

    var_rows.append({
        "Currency": cur,
        **test,
        "Interpretation": (
            "Reject 5% unconditional coverage at 5% level"
            if np.isfinite(test["p_value"]) and test["p_value"] < 0.05
            else "Do not reject 5% unconditional coverage at 5% level"
            if np.isfinite(test["p_value"])
            else "Insufficient observations"
        )
    })

var_diagnostic = pd.DataFrame(var_rows)
display(var_diagnostic)
var_diagnostic.to_csv(TABLE_DIR / "12C_corrected_var_kupiec_diagnostic.csv", index=False)

## Step 9 — Stability audit and explicit quality gate

The final checks distinguish data/timing validity from model performance. A model can pass the data-alignment checks and still perform poorly; conversely, a good-looking score does not excuse a timing error.

In [ ]:
pred = predictions_corrected.copy()
pred["Raw_to_Anchor_Ratio"] = (
    pred["Raw_Selected_Model_Forecast"].abs()
    / pred["Robust_Anchor"].abs().clip(lower=EPS)
)
pred["Final_to_Anchor_Ratio"] = (
    pred["Regime_Robust_Forecast"].abs()
    / pred["Robust_Anchor"].abs().clip(lower=EPS)
)

stability_rows = []
for (cur, h), g in pred.groupby(["Currency", "Horizon"]):
    stability_rows.append({
        "Currency": cur,
        "Horizon": h,
        "Observations": len(g),
        "Stress_Active_Rate_Percent": 100 * g["Stress_Active"].mean(),
        "Guardrail_Active_Rate_Percent": 100 * g["Guardrail_Active"].mean(),
        "Max_Raw_Forecast_to_Anchor": g["Raw_to_Anchor_Ratio"].max(),
        "P99_Raw_Forecast_to_Anchor": g["Raw_to_Anchor_Ratio"].quantile(0.99),
        "Max_Final_Forecast_to_Anchor": g["Final_to_Anchor_Ratio"].max(),
        "Nonpositive_Final_Forecasts": int((g["Regime_Robust_Forecast"] <= 0).sum()),
        "Nonfinite_Final_Forecasts": int((~np.isfinite(g["Regime_Robust_Forecast"])).sum())
    })

stability_audit = pd.DataFrame(stability_rows).sort_values(["Currency", "Horizon"])
display(stability_audit)
stability_audit.to_csv(TABLE_DIR / "12C_corrected_volatility_stability_audit.csv", index=False)

holdout_isolated = (
    df["Date"].max() < pd.Timestamp("2024-01-01")
    and candidate_corrected["Forecast_Origin"].max() < pd.Timestamp("2024-01-01")
    and predictions_corrected["Forecast_Origin"].max() < pd.Timestamp("2024-01-01")
)
target_aligned = n_target_mismatches == 0
strategy_selection_past_only = bool(
    (selections_corrected["Inner_Year"] < selections_corrected["Outer_Year"]).all()
)
forecast_values_valid = bool(
    np.isfinite(predictions_corrected["Regime_Robust_Forecast"]).all()
    and (predictions_corrected["Regime_Robust_Forecast"] > 0).all()
    and np.isfinite(predictions_corrected["EWMA_Benchmark"]).all()
    and (predictions_corrected["EWMA_Benchmark"] > 0).all()
)
matched_origins_available = bool(
    (coverage_audit.loc[coverage_audit["Year"].between(2018, 2023), "Rows_Matched_All_Models"] > 0).all()
)
extreme_raw_ratio_present = bool((stability_audit["Max_Raw_Forecast_to_Anchor"] > 1000).any())

quality_gate = pd.DataFrame([
    {
        "Check": "2024–2025 holdout isolation",
        "Status": "PASS" if holdout_isolated else "FAIL",
        "Evidence": f"Maximum outer forecast date: {predictions_corrected['Forecast_Origin'].max().date()}"
    },
    {
        "Check": "Realized-variance target alignment",
        "Status": "PASS" if target_aligned else "FAIL",
        "Evidence": f"{n_target_mismatches} mismatches after independent reconstruction"
    },
    {
        "Check": "Strategy selection uses prior year only",
        "Status": "PASS" if strategy_selection_past_only else "FAIL",
        "Evidence": f"{len(selections_corrected)} outer task-year selections checked"
    },
    {
        "Check": "Matched-origin comparison",
        "Status": "PASS" if matched_origins_available else "REVIEW",
        "Evidence": "All strategy candidates scored on complete matched origins within each currency/horizon/year"
    },
    {
        "Check": "Forecast values finite and positive",
        "Status": "PASS" if forecast_values_valid else "FAIL",
        "Evidence": "Corrected robust forecasts and EWMA benchmark checked"
    },
    {
        "Check": "Extreme raw model forecasts",
        "Status": "REVIEW" if extreme_raw_ratio_present else "PASS",
        "Evidence": "REVIEW means raw candidate forecasts exceed 1,000x anchor in at least one task; inspect stability audit"
    },
    {
        "Check": "Statistical performance",
        "Status": "REPORT, DO NOT FORCE",
        "Evidence": f"{int(pooled_results['Beats_EWMA_QLIKE'].sum())}/12 pooled QLIKE wins; {int(pooled_results['Significant_QLIKE_Improvement_5pct'].sum())}/12 significant improvements"
    }
])

display(quality_gate)
quality_gate.to_csv(TABLE_DIR / "12C_corrected_volatility_quality_gate.csv", index=False)

if (quality_gate["Status"] == "FAIL").any():
    print("FINAL STATUS: FAIL — do not use the performance table as final research evidence until the failed check is resolved.")
elif (quality_gate["Status"] == "REVIEW").any():
    print("FINAL STATUS: PASS WITH LIMITATIONS — core alignment checks passed, but review all flagged stability/coverage items before freezing.")
else:
    print("FINAL STATUS: TECHNICAL CHECKS PASSED — interpret performance and significance conservatively.")

## Output files

This notebook writes new `12C_` files and preserves the original Script 12 / 12B outputs.

### Predictions
- `outputs/predictions/12C_corrected_candidate_predictions_2018_2023.csv`
- `outputs/predictions/12C_corrected_regime_robust_volatility_predictions_2019_2023.csv`

### Tables
- `outputs/tables/12C_target_alignment_summary.csv`
- `outputs/tables/12C_ewma_alignment_audit.csv`
- `outputs/tables/12C_matched_origin_coverage.csv`
- `outputs/tables/12C_corrected_regime_robust_volatility_selections.csv`
- `outputs/tables/12C_corrected_regime_robust_volatility_strategy_scores.csv`
- `outputs/tables/12C_corrected_volatility_pooled_results.csv`
- `outputs/tables/12C_corrected_volatility_yearly_results.csv`
- `outputs/tables/12C_corrected_var_kupiec_diagnostic.csv`
- `outputs/tables/12C_corrected_volatility_stability_audit.csv`
- `outputs/tables/12C_corrected_volatility_quality_gate.csv`

### Interpretation rules

- **QLIKE lower than EWMA:** better pooled loss for that task.
- **Positive `QLIKE_Gain_vs_EWMA`:** corrected model's QLIKE is lower than corrected EWMA.
- **Statistically significant improvement:** only report when QLIKE improves and the DM-style p-value is below 0.05; still disclose that 12 tasks are being compared.
- **VaR:** a non-significant Kupiec test does not prove perfect calibration. This is only an unconditional-coverage check.
- **Raw GARCH instability:** guardrails can limit the final forecast, but cannot make an unstable raw candidate model intrinsically reliable.
- Do not access or tune on the 2024–2025 holdout in this notebook.